In [0]:
# ============================================================
# GOLD LAYER - KPI DASHBOARD TABLES - UPDATED FOR CI/CD
# ============================================================

import os
from pyspark.sql import functions as F
from datetime import datetime

# ============================================================
# LOAD CONFIGURATION FROM ENVIRONMENT VARIABLES
# ============================================================

# Get environment (DEV, TEST, PROD) - Default to DEV
try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from widget: {env}")
except:
    env = os.getenv('ENV', 'DEV')
    print(f"📌 Environment from env var: {env}")

# Get storage account and access key from environment (GitHub Secrets)
storage_account_name = os.getenv('STORAGE_ACCOUNT', 'capstonestorage01')
access_key = os.getenv('STORAGE_ACCESS_KEY')

# Set container names based on environment
if env == 'DEV':
    gold_container = 'gold-dev'
elif env == 'TEST':
    gold_container = 'gold-test'
else:  # PROD
    gold_container = 'gold'

# Configure Spark with access key (from GitHub Secrets)
if access_key:
    spark.conf.set(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        access_key
    )
    print("✅ Storage access key configured from GitHub Secrets")
else:
    print("⚠️ WARNING: No access key found! Using Azure AD authentication")

# Dynamic path based on environment
GOLD = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 GOLD Container: {gold_container}")
print(f"📍 GOLD Path: {GOLD}")
print(f"{'='*55}\n")

# ============================================================
# YOUR EXISTING CODE (NO CHANGES NEEDED BELOW!)
# ============================================================

# ── Load Facts ──
fact_txn   = spark.read.format("delta").load(f"{GOLD}/fact_transactions")
fact_deliq = spark.read.format("delta").load(f"{GOLD}/fact_delinquency")

# ── KPI 1: Portfolio Exposure ──
kpi_exposure = fact_txn\
    .groupBy("product_type","risk_segment","year")\
    .agg(
        F.round(F.sum("outstanding_balance"),2).alias("total_outstanding"),
        F.round(F.sum("exposure"),2).alias("total_exposure"),
        F.round(F.avg("utilization_pct"),2).alias("avg_utilization_pct"),
        F.countDistinct("account_id").alias("unique_accounts"),
        F.countDistinct("customer_id").alias("unique_customers")
    )
kpi_exposure.write.format("delta").mode("overwrite")\
    .save(f"{GOLD}/kpi_portfolio_exposure")
print(f"✅ kpi_portfolio_exposure    : {kpi_exposure.count():,} rows")

# ── KPI 2: Revenue Analysis ──
kpi_revenue = fact_txn\
    .groupBy("product_type","year","month")\
    .agg(
        F.round(F.sum("interest_accrued"),2).alias("total_interest_income"),
        F.round(F.sum("fee_amount"),2).alias("total_fee_income"),
        F.round(F.sum("repayment_amount"),2).alias("total_repayments"),
        F.round(F.sum("disbursement_amount"),2).alias("total_disbursements"),
        F.count("transaction_id").alias("total_transactions")
    )\
    .withColumn("total_revenue",
        F.round(F.col("total_interest_income") + F.col("total_fee_income"),2))
kpi_revenue.write.format("delta").mode("overwrite")\
    .save(f"{GOLD}/kpi_revenue_analysis")
print(f"✅ kpi_revenue_analysis      : {kpi_revenue.count():,} rows")

# ── KPI 3: Delinquency Trends ──
kpi_delinquency = fact_deliq\
    .groupBy("product_type","dpd_bucket","risk_segment")\
    .agg(
        F.countDistinct("account_id").alias("delinquent_accounts"),
        F.round(F.sum("overdue_amount"),2).alias("total_overdue"),
        F.round(F.avg("days_past_due"),2).alias("avg_dpd"),
        F.round(F.avg("overdue_pct_of_limit"),2).alias("avg_overdue_pct")
    )
kpi_delinquency.write.format("delta").mode("overwrite")\
    .save(f"{GOLD}/kpi_delinquency_trends")
print(f"✅ kpi_delinquency_trends    : {kpi_delinquency.count():,} rows")

# ── KPI 4: Customer Risk ──
kpi_customer_risk = fact_txn\
    .groupBy("risk_segment","income_band","age_group")\
    .agg(
        F.countDistinct("customer_id").alias("total_customers"),
        F.round(F.sum("outstanding_balance"),2).alias("total_exposure"),
        F.round(F.avg("utilization_pct"),2).alias("avg_utilization"),
        F.round(F.sum("interest_accrued"),2).alias("total_interest")
    )
kpi_customer_risk.write.format("delta").mode("overwrite")\
    .save(f"{GOLD}/kpi_customer_risk")
print(f"✅ kpi_customer_risk         : {kpi_customer_risk.count():,} rows")

# ── KPI 5: City Performance ──
kpi_city = fact_txn\
    .groupBy("city","product_type","year")\
    .agg(
        F.countDistinct("customer_id").alias("customers"),
        F.round(F.sum("outstanding_balance"),2).alias("total_outstanding"),
        F.round(F.sum("successful_amount"),2).alias("total_successful"),
        F.count("transaction_id").alias("total_transactions")
    )
kpi_city.write.format("delta").mode("overwrite")\
    .save(f"{GOLD}/kpi_city_performance")
print(f"✅ kpi_city_performance      : {kpi_city.count():,} rows")

# ── Final Summary ──
print("\n" + "=" * 55)
print(f"  GOLD LAYER — FULL SUMMARY ({env})")
print("=" * 55)
all_tables = {
    "dim_customer"          : f"{GOLD}/dim_customer",
    "dim_account"           : f"{GOLD}/dim_account",
    "fact_transactions"     : f"{GOLD}/fact_transactions",
    "fact_delinquency"      : f"{GOLD}/fact_delinquency",
    "kpi_portfolio_exposure": f"{GOLD}/kpi_portfolio_exposure",
    "kpi_revenue_analysis"  : f"{GOLD}/kpi_revenue_analysis",
    "kpi_delinquency_trends": f"{GOLD}/kpi_delinquency_trends",
    "kpi_customer_risk"     : f"{GOLD}/kpi_customer_risk",
    "kpi_city_performance"  : f"{GOLD}/kpi_city_performance",
}
for name, path in all_tables.items():
    count = spark.read.format("delta").load(path).count()
    print(f"  ✅ {name:<30} {count:>10,} rows")
print("=" * 55)
print(f"  ✅ GOLD LAYER COMPLETE! ({env})")
print("=" * 55)

✅ kpi_portfolio_exposure    : 36 rows
✅ kpi_revenue_analysis      : 144 rows
✅ kpi_delinquency_trends    : 27 rows
✅ kpi_customer_risk         : 36 rows
✅ kpi_city_performance      : 84 rows

  GOLD LAYER — FULL SUMMARY
  ✅ dim_customer                      100,000 rows
  ✅ dim_account                       150,000 rows
  ✅ fact_transactions                 974,303 rows
  ✅ fact_delinquency                  200,000 rows
  ✅ kpi_portfolio_exposure                 36 rows
  ✅ kpi_revenue_analysis                  144 rows
  ✅ kpi_delinquency_trends                 27 rows
  ✅ kpi_customer_risk                      36 rows
  ✅ kpi_city_performance                   84 rows
  ✅ GOLD LAYER COMPLETE!
